# FFA-MPDV + SegFormer Deepfake Detection (FaceForensics++ First)

This notebook is the FaceForensics++-first training variant. It prioritizes FF++ frame-style folder structures and keeps the same robust fallback behavior for other fake/real image datasets.

In [ ]:
# Top-level runtime controls for Kaggle
# Set True for a quick sanity run; False for full training.
SMOKE_TEST = True

# Dataset profile for discovery logic: 'ffpp' (default) or 'generic'
DATASET_PROFILE = 'ffpp'

# Optional: set a specific dataset root in Kaggle.
# Example for FF++ frames mirror: '/kaggle/input/faceforensicspp-frames'
DATASET_ROOT = ''

# Full FF++ from official scripts usually yields video-heavy structure.
# For Kaggle training this notebook expects images/frames; enable only limited extraction if needed.
ENABLE_VIDEO_FRAME_EXTRACTION = False
MAX_VIDEOS_TO_EXTRACT = 200
FRAMES_PER_VIDEO = 8

# Optional training profile note for your own tracking
RUN_TAG = 'ffa_mpdv_segformer_ffpp_first'

## Section 1: Environment and Imports
### Block 1.1 - Load required libraries and utilities

In [ ]:
import os
import sys
import random
import math
import hashlib
import warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

import matplotlib
from IPython import get_ipython

# Force notebook-friendly plotting. Use non-GUI backend when not in notebook.
_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic('matplotlib', 'inline')
else:
    matplotlib.use('Agg')

import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    CV2_AVAILABLE = False

try:
    import timm
    TIMM_AVAILABLE = True
except Exception:
    TIMM_AVAILABLE = False

print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('OpenCV available:', CV2_AVAILABLE)
print('timm available:', TIMM_AVAILABLE)
print('Matplotlib backend:', matplotlib.get_backend())

## Section 2: Configuration and Reproducibility
### Block 2.1 - Define config, seeds, and runtime device

In [ ]:
@dataclass
class CFG:
    seed: int = 42
    img_size: int = 256
    batch_size: int = 16
    num_workers: int = 2
    epochs: int = 5
    lr: float = 1e-4
    weight_decay: float = 1e-4
    use_mse_loss: bool = False
    val_size: float = 0.2
    threshold: float = 0.5

    # Run controls
    smoke_test: bool = bool(globals().get('SMOKE_TEST', False))
    smoke_max_train: int = 256
    smoke_max_val: int = 128

    # SegFormer integration
    use_segformer: bool = True
    segformer_variant: str = 'mit_b0'
    segformer_pretrained: bool = True
    segformer_trainable: bool = False

    # Optional manual dataset root
    manual_data_root: str = str(globals().get('DATASET_ROOT', '')).strip()

    # Dataset profile: ffpp first, or generic keyword-only discovery
    dataset_profile: str = str(globals().get('DATASET_PROFILE', 'ffpp')).strip().lower()

    # Optional conversion path if only videos are available
    enable_video_frame_extraction: bool = bool(globals().get('ENABLE_VIDEO_FRAME_EXTRACTION', False))
    max_videos_to_extract: int = int(globals().get('MAX_VIDEOS_TO_EXTRACT', 200))
    frames_per_video: int = int(globals().get('FRAMES_PER_VIDEO', 8))

cfg = CFG()


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(cfg.seed)
is_kaggle = Path('/kaggle/input').exists() or ('KAGGLE_URL_BASE' in os.environ)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if cfg.smoke_test:
    cfg.epochs = min(cfg.epochs, 1)
    cfg.batch_size = min(cfg.batch_size, 8)

# Use single-worker loading on Kaggle and Windows for stability.
if is_kaggle or os.name == 'nt':
    cfg.num_workers = 0

print('Device:', device)
print('Kaggle environment:', is_kaggle)
print('Smoke test mode:', cfg.smoke_test)
print('SegFormer enabled:', cfg.use_segformer)
print('Dataset profile:', cfg.dataset_profile)
print('Video frame extraction enabled:', cfg.enable_video_frame_extraction)

## Section 3: Dataset Discovery
### Block 3.1 - Auto-discover fake/real image paths and build dataframe

In [ ]:
# Kaggle-friendly dataset root discovery (FF++-first variant)
candidate_roots = []
if cfg.manual_data_root:
    candidate_roots.append(Path(cfg.manual_data_root))

candidate_roots.extend([
    Path('/kaggle/input'),
    Path('/kaggle/working'),
    Path.cwd(),
])

# Keep order while removing duplicates
seen = set()
candidate_roots = [p for p in candidate_roots if not (str(p) in seen or seen.add(str(p)))]

img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
video_exts = {'.mp4', '.avi', '.mov', '.mkv'}

# Generic keywords
fake_keywords = ['fake', 'deepfake', 'forged', 'manipulated', 'faceswap', 'synthetic', 'generated']
real_keywords = ['real', 'authentic', 'original', 'pristine', 'genuine', 'youtube']

# FaceForensics++ family names
ffpp_fake_names = {
    'deepfakes', 'face2face', 'faceswap', 'neuraltextures',
    'face_shifter', 'fsgan', 'deepfakedetection', 'deepfakedetection_original', 'fake'
}
ffpp_real_names = {'original', 'originals', 'youtube', 'real'}


def is_image(p: Path):
    return p.suffix.lower() in img_exts


def is_video(p: Path):
    return p.suffix.lower() in video_exts


def classify_dir_name(name: str, profile: str = 'ffpp'):
    low = name.lower().replace('-', '_').replace(' ', '_')

    # FF++-specific priority mapping
    if profile == 'ffpp':
        if low in ffpp_fake_names or any(k in low for k in ffpp_fake_names):
            return 0
        if low in ffpp_real_names or any(k in low for k in ffpp_real_names):
            return 1

    # Generic fallback mapping
    if any(k in low for k in fake_keywords):
        return 0
    if any(k in low for k in real_keywords):
        return 1

    return None


def quick_file_sig(path: str):
    try:
        with open(path, 'rb') as f:
            b = f.read(65536)
        return hashlib.md5(b).hexdigest()
    except Exception:
        return None


def collect_samples(root: Path):
    rows = []
    for d, _, files in os.walk(root):
        dpath = Path(d)
        label = classify_dir_name(dpath.name, cfg.dataset_profile)
        if label is None:
            continue
        for f in files:
            p = dpath / f
            if is_image(p):
                rows.append((str(p), label))
    return rows


def collect_ffpp_videos(root: Path):
    rows = []
    for d, _, files in os.walk(root):
        dpath = Path(d)
        label = classify_dir_name(dpath.name, cfg.dataset_profile)
        if label is None:
            continue
        for f in files:
            p = dpath / f
            if is_video(p):
                rows.append((str(p), label))
    return rows


def extract_frames_from_videos(video_rows):
    if not CV2_AVAILABLE:
        raise RuntimeError('OpenCV (cv2) is required for video frame extraction. Please install cv2 or provide frame dataset.')

    cache_root = Path('/kaggle/working/ffpp_frames_cache' if is_kaggle else './ffpp_frames_cache').resolve()
    (cache_root / 'real').mkdir(parents=True, exist_ok=True)
    (cache_root / 'fake').mkdir(parents=True, exist_ok=True)

    limit = min(len(video_rows), cfg.max_videos_to_extract)
    print(f'Extracting frames from {limit} videos (max).')

    for idx, (vpath, label) in enumerate(video_rows[:limit]):
        cap = cv2.VideoCapture(vpath)
        if not cap.isOpened():
            continue

        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release()
            continue

        cls_dir = cache_root / ('real' if label == 1 else 'fake')
        picks = np.linspace(0, max(total - 1, 0), num=max(1, cfg.frames_per_video), dtype=int)

        base_name = Path(vpath).stem
        for frame_i, fidx in enumerate(picks):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fidx))
            ok, frame = cap.read()
            if not ok or frame is None:
                continue
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            out = cls_dir / f'{base_name}_f{frame_i:02d}.jpg'
            cv2.imwrite(str(out), cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

        cap.release()
        if (idx + 1) % 25 == 0:
            print(f'Processed {idx + 1}/{limit} videos')

    return collect_samples(cache_root)


def generate_smoke_data_if_needed():
    smoke_root = Path('/kaggle/working/smoke_data' if is_kaggle else './smoke_data').resolve()
    for cls_name in ['fake', 'real']:
        d = smoke_root / cls_name
        d.mkdir(parents=True, exist_ok=True)
        for i in range(40):
            arr = np.random.randint(0, 255, size=(cfg.img_size, cfg.img_size, 3), dtype=np.uint8)
            Image.fromarray(arr).save(d / f'{cls_name}_{i:03d}.png')
    print('Generated synthetic smoke dataset at:', smoke_root)
    return collect_samples(smoke_root)


all_rows = []
for root in candidate_roots:
    if root.exists():
        all_rows.extend(collect_samples(root))

# If frames are not found, optionally build frame cache from FF++ videos.
if len(all_rows) == 0 and cfg.enable_video_frame_extraction:
    all_videos = []
    for root in candidate_roots:
        if root.exists():
            all_videos.extend(collect_ffpp_videos(root))
    if len(all_videos) > 0:
        print(f'No frame dataset found. Found {len(all_videos)} videos; building frame cache...')
        all_rows = extract_frames_from_videos(all_videos)

if len(all_rows) == 0 and cfg.smoke_test:
    all_rows.extend(generate_smoke_data_if_needed())

if len(all_rows) == 0:
    existing = [str(p) for p in candidate_roots if p.exists()]
    raise RuntimeError(
        'No labeled fake/real image directories found. '
        'For official FF++ scripts, first create an image/frame dataset, or set ENABLE_VIDEO_FRAME_EXTRACTION=True for limited extraction. '
        f'Checked roots: {existing}'
    )

# Deduplicate by lightweight file signature to reduce leakage risks
dedup_rows = []
seen_sig = set()
for path, label in all_rows:
    sig = quick_file_sig(path)
    key = (sig, label) if sig is not None else (path, label)
    if key in seen_sig:
        continue
    seen_sig.add(key)
    dedup_rows.append((path, label))

df = pd.DataFrame(dedup_rows, columns=['path', 'label']).drop_duplicates()

# Ensure both classes exist in smoke mode to avoid degenerate metrics
if cfg.smoke_test and len(df['label'].unique()) < 2:
    warnings.warn('Only one class detected; generating synthetic samples for the missing class in smoke mode.')
    smoke_rows = generate_smoke_data_if_needed()
    df = pd.concat([df, pd.DataFrame(smoke_rows, columns=['path', 'label'])], ignore_index=True).drop_duplicates()

print('Total discovered images:', len(df))
print(df['label'].value_counts().to_dict())
df.head()

## Section 4: Data Splitting
### Block 4.1 - Create stratified train/validation partitions

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=cfg.val_size,
    random_state=cfg.seed,
    stratify=df['label']
)

if cfg.smoke_test:
    train_df = train_df.groupby('label', group_keys=False).head(max(1, cfg.smoke_max_train // 2)).reset_index(drop=True)
    val_df = val_df.groupby('label', group_keys=False).head(max(1, cfg.smoke_max_val // 2)).reset_index(drop=True)

print('Train:', len(train_df), 'Val:', len(val_df))
print('Train label distribution:', train_df['label'].value_counts().to_dict())
print('Val label distribution:', val_df['label'].value_counts().to_dict())

## Section 5: Input Pipeline
### Block 5.1 - Define transforms, dataset class, and dataloaders

In [ ]:
train_tfms = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=0, shear=8, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

val_tfms = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


class DeepfakeImageDataset(Dataset):
    def __init__(self, frame_df: pd.DataFrame, transform=None, img_size=256):
        self.df = frame_df.reset_index(drop=True)
        self.transform = transform
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row.path).convert('RGB')
        except (FileNotFoundError, UnidentifiedImageError, OSError) as e:
            warnings.warn(f'Failed to load image {row.path}: {e}')
            img = Image.new('RGB', (self.img_size, self.img_size), color=(127, 127, 127))

        if self.transform is not None:
            img = self.transform(img)

        label = torch.tensor(float(row.label), dtype=torch.float32)
        return img, label


train_ds = DeepfakeImageDataset(train_df, train_tfms, img_size=cfg.img_size)
val_ds = DeepfakeImageDataset(val_df, val_tfms, img_size=cfg.img_size)

pin_mem = device.type == 'cuda'
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=pin_mem)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=pin_mem)

print('Batches -> train:', len(train_loader), 'val:', len(val_loader))

## Section 6: Feature Extraction Modules
### Block 6.1 - Build Meso4 backbone, FPN fusion, and spatial attention

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class Meso4Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Sequential(ConvBlock(3, 8, 3, 1), nn.MaxPool2d(2))
        self.c2 = nn.Sequential(ConvBlock(8, 8, 5, 2), nn.MaxPool2d(2))
        self.c3 = nn.Sequential(ConvBlock(8, 16, 5, 2), nn.MaxPool2d(2))
        self.c4 = nn.Sequential(ConvBlock(16, 16, 5, 2), nn.MaxPool2d(4))

    def forward(self, x):
        f1 = self.c1(x)
        f2 = self.c2(f1)
        f3 = self.c3(f2)
        f4 = self.c4(f3)
        return f2, f3, f4

class FPNFusion(nn.Module):
    def __init__(self, channels=(8, 16, 16), out_ch=32):
        super().__init__()
        self.l2 = nn.Conv2d(channels[0], out_ch, 1)
        self.l3 = nn.Conv2d(channels[1], out_ch, 1)
        self.l4 = nn.Conv2d(channels[2], out_ch, 1)
        self.smooth = ConvBlock(out_ch, out_ch, 3, 1)

    def forward(self, f2, f3, f4):
        p4 = self.l4(f4)
        p3 = self.l3(f3) + F.interpolate(p4, size=f3.shape[-2:], mode='nearest')
        p2 = self.l2(f2) + F.interpolate(p3, size=f2.shape[-2:], mode='nearest')
        return self.smooth(p2)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        attn = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * attn

## Section 7: Hybrid Model Assembly
### Block 7.1 - Define capsule routing and FFAMPDV network

In [ ]:
class CapsuleLayer(nn.Module):
    def __init__(self, in_dim, num_caps=8, cap_dim=16, routing_iters=3):
        super().__init__()
        self.num_caps = num_caps
        self.cap_dim = cap_dim
        self.routing_iters = routing_iters
        self.proj = nn.Linear(in_dim, num_caps * cap_dim)

    @staticmethod
    def squash(s, dim=-1, eps=1e-8):
        sq_norm = (s ** 2).sum(dim=dim, keepdim=True)
        scale = sq_norm / (1.0 + sq_norm)
        return scale * s / torch.sqrt(sq_norm + eps)

    def forward(self, x):
        u = self.proj(x).view(x.size(0), self.num_caps, self.cap_dim)
        b = torch.zeros(x.size(0), self.num_caps, 1, device=x.device)

        for _ in range(self.routing_iters):
            c = torch.softmax(b, dim=1)
            s = (c * u).sum(dim=1, keepdim=True)
            v = self.squash(s, dim=-1)
            b = b + (u * v).sum(dim=-1, keepdim=True)

        v = v.squeeze(1)
        return v


class SegformerFeatureExtractor(nn.Module):
    def __init__(self, model_name='mit_b0', pretrained=True, trainable=False, out_dim=64):
        super().__init__()
        self.using_timm = False
        self.fallback_dim = 32

        if TIMM_AVAILABLE:
            candidate_names = [model_name, 'mit_b0', 'hf_hub:nvidia/mit-b0']
            last_err = None
            for cand in candidate_names:
                try:
                    self.backbone = timm.create_model(cand, pretrained=pretrained, num_classes=0, global_pool='avg')
                    self.using_timm = True
                    feat_dim = self.backbone.num_features
                    print('Using SegFormer backbone:', cand)
                    break
                except Exception as e:
                    last_err = e

            if not self.using_timm:
                warnings.warn(f'SegFormer init failed ({last_err}); using lightweight fallback branch.')
                self.backbone = nn.Sequential(
                    nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
                    nn.ReLU(inplace=True),
                    nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
                    nn.ReLU(inplace=True),
                    nn.AdaptiveAvgPool2d(1),
                )
                feat_dim = self.fallback_dim
        else:
            warnings.warn('timm not available; using lightweight fallback branch instead of SegFormer.')
            self.backbone = nn.Sequential(
                nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
                nn.ReLU(inplace=True),
                nn.AdaptiveAvgPool2d(1),
            )
            feat_dim = self.fallback_dim

        if self.using_timm and not trainable:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.proj = nn.Linear(feat_dim, out_dim)

    def forward(self, x):
        if self.using_timm:
            f = self.backbone(x)
        else:
            f = self.backbone(x).flatten(1)
        return self.proj(f)


class FFAMPDVNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        self.backbone = Meso4Backbone()
        self.fpn = FPNFusion(channels=(8, 16, 16), out_ch=32)
        self.sattn = SpatialAttention()
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.meso_head = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )

        self.caps = CapsuleLayer(in_dim=32, num_caps=8, cap_dim=16, routing_iters=3)

        self.segformer_branch = None
        segformer_dim = 0
        if cfg.use_segformer:
            self.segformer_branch = SegformerFeatureExtractor(
                model_name=cfg.segformer_variant,
                pretrained=cfg.segformer_pretrained,
                trainable=cfg.segformer_trainable,
                out_dim=64,
            )
            segformer_dim = 64

        self.classifier = nn.Sequential(
            nn.Linear(64 + 16 + segformer_dim, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        f2, f3, f4 = self.backbone(x)
        fused = self.fpn(f2, f3, f4)
        fused = self.sattn(fused)

        g = self.pool(fused).flatten(1)
        meso_feat = self.meso_head(g)
        cap_feat = self.caps(g)

        feats = [meso_feat, cap_feat]
        if self.segformer_branch is not None:
            seg_feat = self.segformer_branch(x)
            feats.append(seg_feat)

        feat = torch.cat(feats, dim=1)
        logit = self.classifier(feat).squeeze(1)
        return logit


model = FFAMPDVNet(cfg).to(device)
print('Model parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## Section 8: Optimization and Epoch Logic
### Block 8.1 - Configure loss, optimizer, AMP scaler, and run_epoch

In [ ]:
criterion = nn.MSELoss() if cfg.use_mse_loss else nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))


def _safe_classification_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        'acc': float('nan'),
        'precision': float('nan'),
        'recall': float('nan'),
        'f1': float('nan'),
        'roc_auc': float('nan'),
        'pr_auc': float('nan'),
    }

    if y_true.size == 0:
        return metrics, y_pred

    metrics['acc'] = accuracy_score(y_true, y_pred)
    metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
    metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
    metrics['f1'] = f1_score(y_true, y_pred, zero_division=0)

    if len(np.unique(y_true)) > 1 and len(np.unique(y_prob)) > 1:
        metrics['roc_auc'] = roc_auc_score(y_true, y_prob)
        metrics['pr_auc'] = average_precision_score(y_true, y_prob)

    return metrics, y_pred


def run_epoch(loader, train_mode=True):
    model.train(train_mode)
    losses = []
    y_true = []
    y_prob = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(images)
                if cfg.use_mse_loss:
                    probs_for_loss = torch.sigmoid(logits)
                    loss = criterion(probs_for_loss, labels)
                else:
                    loss = criterion(logits, labels)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()

                if torch.isnan(loss) or torch.isinf(loss):
                    warnings.warn('NaN/Inf loss detected. Skipping optimizer step for this batch.')
                else:
                    scaler.step(optimizer)
                    scaler.update()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        y_prob.extend(probs.tolist())
        y_true.extend(labels.detach().cpu().numpy().tolist())

        if torch.isnan(loss) or torch.isinf(loss):
            losses.append(float('nan'))
        else:
            losses.append(loss.item())

    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob)

    if len(losses) == 0:
        raise RuntimeError('No batches were processed. Check dataset and dataloader settings.')

    metrics_core, y_pred = _safe_classification_metrics(y_true, y_prob, threshold=cfg.threshold)
    metrics = {
        'loss': float(np.nanmean(losses)),
        **metrics_core,
    }
    return metrics, y_true, y_prob, y_pred

## Section 9: Training
### Block 9.1 - Execute epoch loop and keep best checkpoint

In [ ]:
history = []
best_val_auc = -1.0
best_state = None

checkpoint_dir = Path('/kaggle/working/checkpoints' if is_kaggle else './checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(1, cfg.epochs + 1):
    tr_metrics, _, _, _ = run_epoch(train_loader, train_mode=True)
    va_metrics, y_true_val, y_prob_val, y_pred_val = run_epoch(val_loader, train_mode=False)

    row = {'epoch': epoch}
    row.update({f'train_{k}': v for k, v in tr_metrics.items()})
    row.update({f'val_{k}': v for k, v in va_metrics.items()})
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {tr_metrics['loss']:.4f} auc {tr_metrics['roc_auc']:.4f} | "
        f"val loss {va_metrics['loss']:.4f} auc {va_metrics['roc_auc']:.4f} f1 {va_metrics['f1']:.4f}"
    )

    current_auc = va_metrics['roc_auc']
    if not math.isnan(current_auc) and current_auc > best_val_auc:
        best_val_auc = current_auc
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

        best_ckpt = {
            'epoch': epoch,
            'best_val_auc': float(best_val_auc),
            'state_dict': best_state,
            'config': cfg.__dict__,
        }
        best_ckpt_path = checkpoint_dir / 'best_checkpoint.pth'
        torch.save(best_ckpt, best_ckpt_path)
        print('Saved best checkpoint:', best_ckpt_path)

history_df = pd.DataFrame(history)
history_df.tail()

## Section 10: Validation and Visualization
### Block 10.1 - Compute final metrics, confusion matrix, ROC and PR curves

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

final_metrics, y_true_val, y_prob_val, y_pred_val = run_epoch(val_loader, train_mode=False)
print('Final validation metrics:', final_metrics)

cm = confusion_matrix(y_true_val, y_pred_val)
print('Confusion matrix:\n', cm)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

if len(np.unique(y_true_val)) > 1 and len(np.unique(y_prob_val)) > 1:
    RocCurveDisplay.from_predictions(y_true_val, y_prob_val, ax=axs[0])
    axs[0].set_title('ROC Curve')

    PrecisionRecallDisplay.from_predictions(y_true_val, y_prob_val, ax=axs[1])
    axs[1].set_title('Precision-Recall Curve')
else:
    axs[0].text(0.5, 0.5, 'ROC not available\n(single class or constant probs)', ha='center', va='center')
    axs[1].text(0.5, 0.5, 'PR not available\n(single class or constant probs)', ha='center', va='center')
    axs[0].set_title('ROC Curve')
    axs[1].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.show()

# Optional saved copy for Kaggle output artifacts.
plot_dir = Path('/kaggle/working' if is_kaggle else './outputs')
plot_dir.mkdir(parents=True, exist_ok=True)
plot_path = plot_dir / 'validation_curves.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
print('Saved plot:', plot_path)

## Section 11: Model Export
### Block 11.1 - Save trained artifact to .pth for Kaggle output

In [ ]:
artifact = {
    'model_name': 'FFA-MPDV-inspired-with-SegFormer-FFPP-first',
    'model_class': 'FFAMPDVNet',
    'state_dict': model.state_dict(),
    'config': cfg.__dict__,
    'history': history,
    'final_metrics': final_metrics,
    'notes': 'Saved as .pth only (no .pkl). Requires FFAMPDVNet definition for loading.',
}

output_dir = Path('/kaggle/output' if is_kaggle else './outputs')
output_dir.mkdir(parents=True, exist_ok=True)
out_path = output_dir / 'ffa_mpdv_segformer_ffpp_first_deepfake_detector.pth'

torch.save(artifact, out_path)
print('Saved:', out_path)

# Optional secondary copy in /kaggle/working for quick inspection in-session
if is_kaggle:
    working_copy = Path('/kaggle/working/ffa_mpdv_segformer_ffpp_first_deepfake_detector.pth')
    torch.save(artifact, working_copy)
    print('Working copy:', working_copy)